In [6]:
import wandb
import pandas as pd

# 1. Fetch the run
api = wandb.Api()
run = api.run("as7629-columbia-university/Multimodal-Nanochat/zgycvj5r")

# 2. Get history as a DataFrame
history_df = run.history()

# 3. Calculate the cumulative sum (first 21 dt entries)
warmup_time = history_df['train/dt'][1:21].sum()
print(warmup_time)
total_hot_time = history_df["total_training_time"].iloc[-1] - warmup_time
print(total_hot_time)
# 4. Save warmup_time and total_hot_time into the run summary
run.summary.update({'warmup_time': float(warmup_time), 'total_hot_time': float(total_hot_time)})



566.5089404582977
289.01555609703064


In [2]:
import wandb
import pandas as pd

# Initialize API
api = wandb.Api()

# Configuration
project_path = "as7629-columbia-university/Multimodal-Nanochat"
group_name = "data_pipeline_tuning_offline" # Replace with your group name

# 1. Fetch all runs in the project with the specified group
runs = api.runs(project_path, filters={"group": group_name})

print(f"Found {len(runs)} runs in group '{group_name}'...")

for run in runs:
    # 2. Get history (only fetch necessary keys to improve performance)
    history_df = run.history(keys=["train/mfu", "train/tok_per_sec", "_step"])

    # 3. Filter for steps 20-30 (inclusive)
    # Using _step ensures we target the actual training steps
    hot_zone = history_df[(history_df["_step"] >= 20) & (history_df["_step"] <= 30)]

    if not hot_zone.empty:
        # Calculate averages
        avg_mfu = hot_zone["train/mfu"].mean()
        avg_tok = hot_zone["train/tok_per_sec"].mean()

        # 4. Update the run summary
        run.summary["avg_hot_mfu"] = float(avg_mfu)
        run.summary["avg_hot_tok_per_sec"] = float(avg_tok)
        
        # Persist changes to the W&B server
        run.summary.update()
        print(f"Updated run {run.id}: avg_hot_mfu={avg_mfu:.4f}")
    else:
        print(f"Skipping run {run.id}: No data found for steps 20-30.")

Found 2 runs in group 'data_pipeline_tuning_offline'...
Updated run tzqkgfas: avg_hot_mfu=24.6549
Updated run hx9jqg2u: avg_hot_mfu=24.7993


In [6]:
import wandb

def duplicate_wandb_run(original_run_path, new_project, new_group, new_name):
    """
    Duplicates an existing wandb run to a new project/group/name.
    
    :param original_run_path: String in format "entity/project/run_id"
    :param new_project: Target project name
    :param new_group: New group name for the duplicate
    :param new_name: New run name
    """
    api = wandb.Api()
    
    # 1. Fetch the original run
    old_run = api.run(original_run_path)
    
    print(f"Duplicating run: {old_run.name} ({old_run.id})")

    # 2. Initialize the new run
    # Note: We pass the old config directly to the new run
    new_run = wandb.init(
        project=new_project,
        group=new_group,
        name=new_name,
        config=old_run.config,
        resume="allow"
    )

    # 3. Copy Summary Metrics
    # This updates the dashboard with the final reported results of the old run
    summary_dict = dict(old_run.summary)
    new_run.summary.update(summary_dict)

    # 4. Copy History (Time-series data)
    # Be aware: For very long training runs, this loop can take time.
    # We iterate through the historical data and log it to the new run.
    history = old_run.history()
    for _, row in history.iterrows():
        # Clean up NaNs which can break W&B logging
        log_data = row.dropna().to_dict()
        new_run.log(log_data)

    print(f"Successfully created new run: {new_run.url}")
    new_run.finish()

# Usage
if __name__ == "__main__":
    duplicate_wandb_run(
        original_run_path="as7629-columbia-university/Multimodal-Nanochat/p79t3b33",
        new_project="Multimodal-Nanochat",
        new_group="deviceBS_validation_iteration",
        new_name="iteration500_deviceBS16_bfloat_profileFalse16"
    )

Duplicating run: totalBS32768_lr1e-2 (p79t3b33)


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


TypeError: Object of type SummarySubDict is not JSON serializable

In [ ]:
import wandb

# ---------------------------------------------------------
# 1. Configuration: Replace these with your actual details
# ---------------------------------------------------------
ENTITY = "as7629-columbia-university"          # Your W&B username or team name
PROJECT = "Multimodal-Nanochat"        # The project name
OLD_RUN_ID = "p79t3b33"       # The 8-character ID of the run to duplicate

NEW_RUN_NAME = "iteration500_totalBS16384"
NEW_GROUP_NAME = "deviceBS_validation_iteraction"

# ---------------------------------------------------------
# 2. Fetch the old run data
# ---------------------------------------------------------
api = wandb.Api()
old_run_path = f"{ENTITY}/{PROJECT}/{OLD_RUN_ID}"
print(f"Fetching old run: {old_run_path}...")
old_run = api.run(old_run_path)

# ---------------------------------------------------------
# 3. Initialize the new run
# ---------------------------------------------------------
print(f"Initializing new run '{NEW_RUN_NAME}' in group '{NEW_GROUP_NAME}'...")
wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=NEW_RUN_NAME,
    group=NEW_GROUP_NAME,
    config=old_run.config,      # Duplicate the configuration/hyperparameters
    tags=old_run.tags,          # (Optional) Carry over tags
    notes=old_run.notes         # (Optional) Carry over notes
)

# ---------------------------------------------------------
# 4. Copy the historical metrics over to the new run
# ---------------------------------------------------------
print("Copying historical metrics. This may take a moment depending on run length...")

# scan_history() is the safest way to iterate through all logged metrics
for row in old_run.scan_history():
    # Capture the original step to keep charts perfectly aligned
    step = row.get("_step")
    
    # Strip out internal W&B keys (like _timestamp, _runtime) to avoid conflicts
    cleaned_row = {k: v for k, v in row.items() if not k.startswith("_")}
    
    # Log the cleaned data to the new run
    if cleaned_row:
        wandb.log(cleaned_row, step=step)

# ---------------------------------------------------------
# 5. Finish and sync to the dashboard
# ---------------------------------------------------------
wandb.finish()
print("Success! Your duplicated run has been pushed to the W&B dashboard.")

Fetching old run: as7629-columbia-university/Multimodal-Nanochat/p79t3b33...
Initializing new run 'iteration500_deviceBS16_bfloat_profileFalse16' in group 'deviceBS_validation_iteraction'...


Copying historical metrics. This may take a moment depending on run length...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


total_training_flops,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████
total_training_time,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇██
train/dataloader_fraction,▂▄▃▄▁▂▆▅▅▅▆▅▆▄▅▅█▅█▆▄▅▆▆▅▆▅▅▄▄▄▃▅▆▃▃▆▅▅▅
train/dataloader_wait_ms,▃▃▃▁▂▂▃▃▄▄▄▃▄█▃▄▅▆▄▄▃▅▅▅▃▃▁▁▂▃▄▄▁▃▃▄▃▃▃▃
train/dt,▅▅▃▅▆▆▆▆▄▃▅▅▅▃▄▄▆▁▄▅▄▆▆▄▅▄▅▄▅▅▅▆▇█▅▅▅▅▅▅
train/epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/h2d_transfer_ms,▂▃▂▂▁▅▅▆▅▇▅▆▅▄█▆▅▄▆▃▅▆▄▄▃▃▄▃▆▄▃▆▄▆▃▆█▃▅▅
train/loss,███▇▆▆▅▅▄▄▃▃▃▃▄▃▃▃▃▄▄▃▃▃▂▃▃▃▃▃▂▂▃▃▁▁▃▃▃▃
train/mfu,█▅▄▄▄▆▅▄▆▄▆▅▅▅▄▁▇▅█▆▄▅▃▃▃▄▄▇▃▁█▅▅▄▄▇▄▄▅▆
train/tok_per_sec,▅▅▆▅▅▄█▄▄▄▅▆▅▅▁▃▁▅▆▆▄▄▅▅▅▅▆▅▃▅▆▅▅▁▆▅▅▅▃▅
+5,...


Success! Your duplicated run has been pushed to the W&B dashboard.
